> NOTA: Esta prática tangencia o uso de DL em grandes datasets e grandes modelos, portanto, aqui não utilizaremos as travas de determinismo a nível de GPU comentadas durante as aulas

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from absl import logging

# Define o nível de logging para silenciar os avisos do TF
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

# contas internas em float16: treino cerca de 2x mais rápido na GPU, sem mudar o resultado significativamente (NESTA PRÁTICA)
keras.mixed_precision.set_global_policy('mixed_float16')

# Mini GPT: completando texto, caractere a caractere

Na aula usamos o Transformer para **classificar** um texto. Aqui vamos usar as mesmas peças para **gerar** texto: o modelo recebe um trecho e prevê qual é o próximo caractere. Repetindo essa previsão várias vezes, ele escreve sozinho. É a ideia por trás dos modelos da família GPT, em versão de bolso.

Roteiro: `1.` dados → `2.` vetorização por caractere → `3.` entradas e alvos → `4.` máscara causal → `5.` modelo → `6.` treino → `7.` geração.

> **Antes de começar:** O treino leva cerca de 40 minutos em uma GPU T4. No Colab, ative em `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4`.

> <span style="color:red"> **Sobre o que esperar:** </span>  Este modelo é milhares de vezes menor que um GPT de verdade. Ele aprende a escrever palavras e frases com cara de notícia, mas sem coerência entre uma frase e outra. Todo texto gerado é inventado, inclusive quando cita pessoas e lugares reais. O maior objetivo desta prática é entender o mecanismo programático do GPT.

## Definição da task

O modelo recebe uma sequência de caracteres `x` e deve prever, **em cada posição**, qual é o caractere seguinte. Por isso o alvo `y` é o próprio `x`, deslocado um caractere para a frente:

```
x:  t   e   x   t
y:  e   x   t   o

posição 0: vê "t"     → deve prever "e"
posição 1: vê "te"    → deve prever "x"
posição 2: vê "tex"   → deve prever "t"
posição 3: vê "text"  → deve prever "o"
```

Ou seja, a saída não é uma única previsão por sequência: é uma softmax **para cada posição**. Uma sequência de 4 caracteres rende 4 exemplos de treino de uma vez, cada um com um contexto de tamanho diferente. Na hora de gerar texto, usamos apenas a previsão da última posição.

```
entrada x:       (BATCH, SEQ_LEN)               índices dos caracteres de entrada
alvo y:          (BATCH, SEQ_LEN)               índices dos caracteres seguintes
saída do modelo: (BATCH, SEQ_LEN, VOCAB_SIZE)   uma softmax por posição
```

**O problema:** na self-attention da aula, cada posição enxerga a sequência inteira. A posição 0 veria o `e` logo à frente, que é justamente a resposta que ela deveria prever. O modelo acertaria tudo no treino apenas copiando, é pior que overfitting!

A solução é a **máscara causal**, que impede cada posição de olhar para as que vêm depois dela. Ela é o assunto da seção 4.

## 1. Dados

Vamos usar reportagens da BBC Brasil, publicadas entre 2004 e 2021. Elas fazem parte do dataset XL-Sum (Hasan et al., 2021), criado para a tarefa de resumo de textos. Aqui só nos interessa o corpo das reportagens.

> Licença CC BY-NC-SA 4.0: uso não comercial. Neste material, apenas para fins educacionais.

O arquivo é baixado e extraído apenas se ainda não existir:

In [ ]:
URL = 'https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/data/portuguese_XLSum_v2.0.tar.bz2'

if not os.path.exists('portuguese_train.jsonl'):
    !wget -q {URL}
    !tar -xjf portuguese_XLSum_v2.0.tar.bz2

Cada linha do arquivo é uma reportagem. Usamos uma fatia do treino e uma fatia pequena da validação, já separadas pelo próprio dataset. O `'\n'` no final marca onde cada reportagem termina.

> **Por que 4000?** O arquivo de treino tem 57.402 reportagens, e esse é o teto de `N_TRAIN`. O valor de 4000 (cerca de 28 milhões de caracteres) foi escolhido à mão, para o treino caber no tempo e na memória do Colab gratuito.

> **Fins didáticos:** com essa quantidade de texto, espere frases de baixa coerência. O GPT-2, de 2019, foi treinado com cerca de 40 GB de texto, mais de mil vezes o que usamos aqui, e ainda era um modelo muito maior (e continuava ruim kkkk). O objetivo desta prática é entender o mecanismo; não espere muito.

In [ ]:
N_TRAIN = 4000   # reportagens de treino
N_VAL = 500      # reportagens de validação


def load_corpus(path, n):
    """Lê as n primeiras reportagens de um arquivo .jsonl e retorna um array de strings."""
    df = pd.read_json(path, lines=True, nrows=n)
    return (df['text'] + '\n').to_numpy()


train_corpus = load_corpus('portuguese_train.jsonl', N_TRAIN)
val_corpus = load_corpus('portuguese_val.jsonl', N_VAL)

print(f'Train corpus: {train_corpus.shape}')
print(f'Val corpus: {val_corpus.shape}')
print(f"{'-'*20}\nAmostra | {train_corpus[0][:500]}")

## 2. Vetorização por caractere

Na aula, cada token era uma palavra. Aqui, cada token é um **caractere**. A camada é a mesma `TextVectorization`, com dois ajustes:

- `split='character'`: quebra o texto letra por letra;
- `standardize=None`: mantém maiúsculas, acentos e pontuação, porque o modelo precisa aprender a escrevê-los.

**Vantagens** em relação ao token por palavra:

- vocabulário minúsculo: cerca de 100 símbolos dão conta de quase todo o texto, contra 20 mil palavras no IMDB;
- quase nada fica de fora: qualquer palavra, nome ou número pode ser escrito com esses símbolos, sem cair no `[UNK]`;
- menos parâmetros: a tabela de embeddings e a camada de saída ficam bem menores.

**Desvantagens:**

- um caractere sozinho não carrega significado: o modelo precisa aprender a formar palavras antes de aprender a formar frases;
- as sequências ficam bem mais longas: o mesmo texto vira cerca de 5 vezes mais tokens, então o modelo enxerga menos texto de cada vez e o treino fica mais caro.

In [ ]:
VOCAB_SIZE = 100

vectorizer = keras.layers.TextVectorization(
    standardize=None,
    split='character',
    max_tokens=VOCAB_SIZE,
    ragged=True,  # cada reportagem com o seu próprio tamanho, sem padding
)
vectorizer.adapt(train_corpus)

vocab = np.array(vectorizer.get_vocabulary())  # ordenado do mais para o menos frequente
print(vocab)

Os dois primeiros índices continuam reservados: `''` (padding) e `[UNK]`, que aqui representa os símbolos raros que ficaram fora dos 100 mais frequentes.

> NOTA: Nesta prática, o modelo pode simplesmente prever `[UNK]` como próximo token, pelo simples fato dele estar presente no treinamento. Em modelos modernos, os vetorizadores incluem absolutamente tudo, então acaba não existindo `[UNK]`.

Vetorizamos tudo e emendamos as reportagens em uma única sequência longa de índices (`flat_values`):

> Nota: emendar todas as amostras em um único texto pode parecer prejudicial pela mudança abrupta de assunto. Na realidade, esse efeito é significativamente minimizado justamente por mantermos o `\n` como token. Ele permitirá ao modelo aprender que depois dele o assunto muda. Inclusive, o próprio modelo simplificado poderá utilizar `\n` no meio de uma inferência e mudar completamente de assunto em seguida.

In [ ]:
train_ids = vectorizer(train_corpus).flat_values.numpy()
val_ids = vectorizer(val_corpus).flat_values.numpy()

print(f'Caracteres de treino: {len(train_ids)}')
print(f'Caracteres de validação: {len(val_ids)}')
print(f'Fora do vocabulário ([UNK]): {(train_ids == 1).mean():.3%}')

O caminho de volta, de índices para texto, é só consultar o vocabulário:

In [ ]:
def decode(ids):
    """Converte uma sequência de índices de volta em texto."""
    return ''.join(vocab[ids]) # espaço vazio também é token. Por isso string fechada no join ''.


print(train_ids[:40])
print(decode(train_ids[:40]))

## 3. Entradas e alvos

<span style="color:red"> De onde vêm os rótulos, se ninguém anotou este texto?</span> => <span style="color:green"> Do próprio texto. </span>

O alvo é a mesma sequência da entrada, **deslocada um caractere para a frente**. Para cada posição, a resposta certa é o caractere seguinte:

```
entrada: O  g o v e r n
alvo:    g o v e r n o
```

Cortamos a sequência longa em janelas de `SEQ_LEN + 1` caracteres. Os primeiros `SEQ_LEN` formam a entrada, os últimos `SEQ_LEN` formam o alvo:

> `SEQ_LEN` grande: maior coerência contextual, treino mais lento, modelo maior, mais memória O(N²)

In [ ]:
# quantos caracteres o modelo enxerga de uma vez (centenas de milhares nos modelos de fronteira, até milhão)
# quanto maior, melhor a coerência contextual, mas o custo é O(N²)
SEQ_LEN = 256
BATCH_SIZE = 128 # (aprox. o limite da GPU T4, ~12GB para treinamento)


def get_dataset(ids, shuffle=False):
    """Corta a sequência em janelas e monta um tf.data.Dataset de (entrada, alvo)."""
    # quantas janelas inteiras de SEQ_LEN + 1 caracteres cabem na sequência
    n_windows = len(ids) // (SEQ_LEN + 1)
    # descarta a sobra do final e organiza em uma tabela: uma janela por linha
    windows = ids[:n_windows * (SEQ_LEN + 1)].reshape(n_windows, SEQ_LEN + 1)
    # entrada: tudo menos o último caractere | alvo: tudo menos o primeiro
    x, y = windows[:, :-1], windows[:, 1:]  # alvo: a entrada deslocada em um caractere

    # cada par (entrada, alvo) vira um exemplo do dataset
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    # no treino, embaralha a ordem das janelas
    ds = ds.shuffle(n_windows) if shuffle else ds
    # agrupa em batches (o último, se incompleto, é descartado) e já deixa o próximo preparado
    return ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(1)


train_ds = get_dataset(train_ids, shuffle=True)
val_ds = get_dataset(val_ids)

In [ ]:
for x, y in train_ds:
    print('x.shape:', x.shape)
    print('y.shape:', y.shape)
    print(f"{'-'*20}\nentrada | {decode(x[0, :60].numpy())}")
    print(f'   alvo | {decode(y[0, :60].numpy())}')
    break

Repare que cada janela vale por `SEQ_LEN` exemplos de treino: o modelo faz uma previsão em **cada** posição da sequência, e não só no final.

## 4. Máscara causal

Relembrando a definição da task, existe um problema: na self-attention da aula, cada token olha para **todos** os tokens da frase, inclusive os que vêm depois. Para prever o próximo caractere isso seria colar, pois a resposta está logo à frente.

A solução é a **máscara causal**: cada posição só pode prestar atenção nela mesma e nas anteriores. Na matriz de scores `(s, s)` vista em aula, isso equivale a zerar tudo acima da diagonal:

In [ ]:
tokens = list('texto')
mask = np.tril(np.ones((len(tokens), len(tokens)), dtype='int'))

# linha: quem está olhando | coluna: para quem pode olhar (1 = pode, 0 = bloqueado)
pd.DataFrame(mask, index=tokens, columns=tokens)

No Keras não é preciso montar essa matriz na mão: basta chamar a `MultiHeadAttention` com `use_causal_mask=True` :D

## 5. Modelo

As peças são as mesmas do Transformer da aula:

- `Embedding`: cada caractere vira um vetor de `EMBED_DIM` posições;
- positional encoding com senos e cossenos, somado ao embedding;
- blocos de atenção + feed-forward, com conexões residuais e normalização.

Duas diferenças em relação ao classificador do IMDB:

- a atenção usa a **máscara causal**;
- a saída não é um único número: é uma **softmax com `VOCAB_SIZE` posições para cada caractere da sequência**, a probabilidade de cada símbolo ser o próximo.

> Lembra que na aula o código do `PositionalEncoder` e `TransformerEncoder` eram classes de custom Layer?
>
> O código aqui é mais simples: funções curtas no lugar das classes `SinCosPositionalEncoder` e `TransformerEncoder`. Isso é possível por duas razões. 
> * Não há padding, pois toda janela tem exatamente `SEQ_LEN` caracteres, então não é preciso levar a máscara de padding pela rede. 
> * Há texto de sobra, então o dropout não faz falta.

In [ ]:
def positional_encoding(seq_len, d_model):
    """Matriz (seq_len, d_model) de senos e cossenos, a mesma fórmula da aula."""
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angles = pos / np.power(10000, (2 * (i // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])  # dimensões pares: seno
    angles[:, 1::2] = np.cos(angles[:, 1::2])  # dimensões ímpares: cosseno
    return angles.astype('float32')


def transformer_block(x, embed_dim, num_heads):
    """Atenção causal + feed-forward, cada uma com conexão residual e normalização."""
    attention = keras.layers.MultiHeadAttention(num_heads, key_dim=embed_dim // num_heads)
    x = keras.layers.LayerNormalization()(x + attention(x, x, use_causal_mask=True))

    # transformers modernos utilizam "gelu" ou "SwiGLU", mas vamos de original por enquanto
    # -e não deve fazer grandes diferenças nessas circunstâncias-
    dense = keras.layers.Dense(embed_dim * 4, activation='relu')(x)
    dense = keras.layers.Dense(embed_dim)(dense)
    return keras.layers.LayerNormalization()(x + dense)

In [ ]:
EMBED_DIM = 256 # word embedding (dezenas de milhares em modelos de fronteira)
NUM_HEADS = 4   # cada head de atenção aprende coisas diferentes na mesma camada
NUM_BLOCKS = 6  # layers transformer empilhados (centenas em modelos de fronteira; possivelmente milhares)

inputs = keras.layers.Input(shape=(SEQ_LEN,), dtype='int64')
x = keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM)(inputs)        # => (batch, SEQ_LEN, EMBED_DIM)
x = x * EMBED_DIM ** 0.5 + positional_encoding(SEQ_LEN, EMBED_DIM) # reescalonamento para estabilidade com o PE

for _ in range(NUM_BLOCKS):
    x = transformer_block(x, EMBED_DIM, NUM_HEADS)

# NOTE: dtype='float32' na saída é exigência do mixed precision
outputs = keras.layers.Dense(VOCAB_SIZE, activation='softmax', dtype='float32')(x)  # (batch, SEQ_LEN, VOCAB_SIZE)

model = keras.Model(inputs, outputs)
model.summary()

## 6. Treino

Antes de treinar, a função que usa o modelo para escrever. A cada passo:

1. o modelo recebe o texto escrito até agora;
2. devolve a probabilidade de cada caractere ser o próximo;
3. sorteamos um caractere seguindo essas probabilidades;
4. o caractere sorteado entra no texto, e o ciclo recomeça.

A `temperature` ajusta o sorteio, e será explorada na seção 7. Por enquanto fica em 1, que usa as probabilidades do modelo sem alteração.

In [ ]:
def generate(prompt, n_chars=300, temperature=1.0):
    """Completa o prompt com n_chars caracteres, sorteados um a um."""
    ids = list(vectorizer([prompt]).flat_values.numpy())

    for _ in range(n_chars):
        context = ids[-SEQ_LEN:]  # o modelo só enxerga os últimos SEQ_LEN caracteres (famosa janela de contexto!!)
        x = np.zeros((1, SEQ_LEN), dtype='int64') # prepara um batch vazio no formato esperado
        x[0, :len(context)] = context  # sobra à direita fica com 0: a máscara causal impede que atrapalhe

        probs = model.predict_on_batch(x)[0, len(context) - 1]  # previsão para o próximo caractere
        if temperature == 0:
            ids.append(int(np.argmax(probs)))  # sem sorteio: sempre o caractere mais provável
        else:
            probs = probs.astype('float64') ** (1 / temperature)
            ids.append(np.random.choice(VOCAB_SIZE, p=probs / probs.sum()))

    return decode(ids)

Sem treino nenhum, os pesos são aleatórios e o resultado é ruído:

In [ ]:
print(generate('O governo '))

Agora o treino. A loss é a mesma entropia cruzada de sempre, calculada em cada posição da sequência. A acurácia responde: em quantos por cento das posições o caractere mais provável era o correto?

Ao final de cada época o modelo escreve uma amostra. Vale acompanhar a evolução: primeiro ruído, depois sílabas, palavras e, por fim, frases com pontuação.

In [ ]:
EPOCHS = 20 # na prática, levaria muito mais épocas para convergência, mas não vale a pena pela didática
LEARNING_RATE = 5e-4
WARMUP_STEPS = 1000  # O LR vai começar em 0 e 'aquecer' até LEARNING_RATE ao longo de 1000 batches de treino
CHECKPOINT = 'mini_gpt.weights.h5'

model.compile(
    # AdamW: Adam com weight decay, que impede os pesos de crescerem demais
    # global_clipnorm limita a norma do gradiente inteiro: treino mais estável
    optimizer=keras.optimizers.AdamW(LEARNING_RATE, weight_decay=0.01, global_clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)


def make_callbacks(checkpoint):
    """Monta os callbacks do treino: warmup, amostra de texto e checkpoint dos melhores pesos.
       NOTA: Não usei Earlystopping aqui porque não faz muito sentido. Um modelo desses 
             requeriria muito mais épocas para convergência total, impraticável para estudo.
    """

    def warmup(batch, logs):
        step = int(model.optimizer.iterations)  # isso retorna quantos passos de treino já rodaram
        if step < WARMUP_STEPS:
            model.optimizer.learning_rate = LEARNING_RATE * (step + 1) / WARMUP_STEPS

    # warmup: no início os gradientes do transformer são instáveis,
    #         então começamos com passos pequenos para não explodir
    lr_warmup = keras.callbacks.LambdaCallback(on_train_batch_begin=warmup) # executa warmup() a cada batch rodado

    # escreve uma amostra ao final de cada época
    show_sample = keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: print(f"\n{'-'*20}\n{generate('O governo ', n_chars=200)}\n{'-'*20}")
    )

    # salva os pesos sempre que a val_loss melhora
    # salvar apenas os pesos é uma boa prática (mais fácil de migrar para outros ecossistemas além do keras)
    save_best = keras.callbacks.ModelCheckpoint(checkpoint, save_weights_only=True, save_best_only=True)

    return [lr_warmup, show_sample, save_best]


history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=make_callbacks(CHECKPOINT))
model.load_weights(CHECKPOINT)

In [ ]:
df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    df[[metric, f'val_{metric}']].plot(ax=ax, title=metric, xlabel='epoch', grid=True)
plt.show()

Uma referência para ler a loss: chutando ao acaso entre 100 símbolos, ela seria `ln(100) ≈ 4.6`. Quanto mais baixa, mais certeiro o palpite do modelo sobre o próximo caractere.

## 7. Geração

Com o modelo treinado, é só escolher o começo do texto:

In [ ]:
print(generate('A economia brasileira segue uma tendência ', n_chars=500))

### 7.1 Temperatura

A temperatura controla o quanto o sorteio se arrisca:

- **zero** (0): não há sorteio, o modelo escolhe sempre o caractere mais provável. O texto sai igual a cada execução e tende a entrar em repetição;
- **baixa** (0.5): reforça os caracteres mais prováveis. Texto mais correto, porém repetitivo;
- **1.0**: usa as probabilidades exatamente como o modelo calculou;
- **alta** (1.5): dá mais chance aos caracteres improváveis. Texto mais variado, com mais erros.

In [ ]:
for temperature in [0, 0.5, 1.0, 1.5]:
    print(f"{'-'*20}\ntemperature = {temperature}")
    print(generate('Cientistas descobriram ', temperature=temperature))

### 7.2 Teste Genérico Avulso

Troque o prompt e a temperatura à vontade:

In [ ]:
print(generate('', n_chars=500, temperature=1.))

## 8. Para ir além

O que separa este modelo de um GPT de verdade é principalmente escala:

- **token**: os modelos reais usam pedaços de palavras (subwords), um meio-termo entre caractere e palavra;
- **tamanho**: aqui são cerca de 3 milhões de parâmetros, contra bi/tri nos modelos de fronteira;
- **dados**: aqui são 4 mil reportagens, contra boa parte da internet;
- **contexto**: aqui o modelo enxerga 256 caracteres, contra centenas de milhares de tokens.
- **GPT + RF**: Fizemos aqui a etapa GPT, generative-pretrained-transformer. Um papagaio estocastico glorificado. Depois disso, ainda seria necessário uma etapa de aprendizado por reforço, onde o GPT (já treinado) é alimentado com objetivos amplos (e.g., resolva essa prova matemática; encontre e conserte o bug desse código, resolva essa prova do ENEM, etc etc). Aqui, o modelo deve resolver os objetivos somente por meio de texto. Quando ele consegue, é bonificado, do contrário, penalizado; é mais uma etapa de treinamento longa e pesadíssima. 
- **multi modalidade**: adicionar outros tipos de dados em cima disso tudo. Texto, código (que também é texto), imagens, áudios, etc.
- E claro, centenas de milhares das melhores GPUs do mercado e semanas para completar um treinamento. Harness, guard rails, vieses corretivos, etc etc...

A lógica é a mesma: prever o próximo token, um de cada vez.

> dica de exercício prático: mude `SEQ_LEN`, `NUM_BLOCKS` ou `N_TRAIN` e compare a loss de validação e a qualidade do texto gerado. (mas cuidado para não estorar o limite de computação, basta ir com valores pequenos para uma comparação nítida)

Referências:

- KARPATHY, A. *Let's build GPT: from scratch, in code, spelled out*.
- VASWANI, A. et al. *Attention Is All You Need* (2017).
- HASAN, T. et al. *XL-Sum: Large-Scale Multilingual Abstractive Summarization for 44 Languages* (2021).